In [1]:
from pathlib import Path
import sys
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from feature_analysis import find_high_correlation_pairs
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
 )

sys.path.insert(0, str(Path("..").resolve()))
from backend.registry import register_dataset, register_model

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(Path("..").resolve()))

In [2]:
df = pd.read_csv('brfss_2024.zip', compression='zip')
pd.set_option('display.max_columns', None)

target_col = 'CVDSTRK3'

# Keep only rows where we have a clear stroke status
df = df[(df[target_col] == 1) | (df[target_col] == 2)]

# Recode BRFSS survey values (1=Yes, 2=No) to standard binary (1=Stroke, 0=No Stroke)
df[target_col] = (df[target_col] == 1).astype(int)

print("Amount of rows and features:", df.shape)
df.head(1)

Amount of rows and features: (456218, 301)


,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENM1,PVTRESD1,COLGHOUS,STATERE1,CELPHON1,LADULT1,NUMADULT,RESPSLC1,LANDSEX3,SAFETIME,CTELNUM1,CELLFON5,CADULT1,CELLSEX3,PVTRESD3,CCLGHOUS,CSTATE1,LANDLINE,HHADULT,SEXVAR,GENHLTH,PHYSHLTH,MENTHLTH,POORHLTH,PRIMINS2,PERSDOC3,MEDCOST1,CHECKUP1,EXERANY2,LASTDEN4,RMVTETH4,CVDINFR4,CVDCRHD4,CVDSTRK3,ASTHMA3,ASTHNOW,CHCSCNC1,CHCOCNC1,CHCCOPD3,ADDEPEV3,CHCKDNY2,HAVARTH4,DIABETE4,DIABAGE4,MARITAL,EDUCA,RENTHOM1,NUMHHOL4,NUMPHON4,CPDEMO1C,VETERAN3,EMPLOY1,CHILDREN,INCOME3,PREGNANT,WEIGHT2,HEIGHT3,DEAF,BLIND,DECIDE,DIFFWALK,DIFFDRES,DIFFALON,HADMAM,HOWLONG,CERVSCRN,CRVCLCNC,CRVCLPAP,CRVCLHPV,HADHYST2,HADSIGM4,COLNSIGM,COLNTES1,SIGMTES1,LASTSIG4,COLNCNCR,VIRCOLO1,VCLNTES2,SMALSTOL,STOLTEST,STOOLDN2,BLDSTFIT,SDNATES1,SMOKE100,SMOKDAY2,USENOW3,ECIGNOW3,LCSFIRST,LCSNUMCG,LCSCTSC1,LCSSCNCR,LCSCTWHN,ALCDAY4,AVEDRNK4,DRNK3GE5,MAXDRNKS,FLUSHOT7,FLSHTMY3,IMFVPLA5,PNEUVAC4,HIVTST7,HIVTSTD3,HIVRISK5,PDIABTS1,PREDIAB2,DIABTYPE,INSULIN1,CHKHEMO3,EYEEXAM1,DIABEYE1,DIABEDU1,FEETSORE,ARTHEXER,SHINGLE2,HPVADVC4,HPVADSH1,TETANUS1,CNCRDIFF,CNCRAGE,CNCRTYP2,CSRVTRT3,CSRVDOC1,CSRVSUM,CSRVRTRN,CSRVINST,CSRVINSR,CSRVDEIN,CSRVCLIN,CSRVPAIN,CSRVCTL2,PSATEST1,PSATIME1,PCPSARS2,PSASUGS1,PCSTALK2,CIMEMLO1,CDWORRY,CDDISCU1,CDHOUS1,CDSOCIA1,CAREGIV1,CRGVREL5,CRGVPRB4,CRGVALZD,CRGVNURS,CRGVPER2,CRGVHOU2,CRGVHRS2,CRGVLNG2,ACEDEPRS,ACEDRINK,ACEDRUGS,ACEPRISN,ACEDIVRC,ACEPUNCH,ACEHURT1,ACESWEAR,ACETOUCH,ACETTHEM,ACEHVSEX,ACEADSAF,ACEADNED,LSATISFY,EMTSUPRT,SDLONELY,SDHEMPLY,FOODSTMP,SDHFOOD1,SDHBILLS,SDHUTILS,SDHTRNSP,HOWSAFE1,MARIJAN1,MARJSMOK,MARJEAT,MARJVAPE,MARJDAB,MARJOTHR,USEMRJN4,LASTSMK2,STOPSMK2,MENTCIGS,MENTECIG,HEATTBCO,SSBSUGR2,SSBFRUT3,FIREARM5,GUNLOAD,LOADULK2,RCSGEND1,RCSXBRTH,RCSRLTN2,CASTHDX2,CASTHNO2,SOMALE,SOFEMALE,HADSEX,PFPPRVN4,TYPCNTR9,NOBCUSE8,QSTVER,QSTLANG,HPVDSHT,ICFQSTVR,_METSTAT,_URBSTAT,MSCODE,_STSTR,_STRWT,_RAWRAKE,_WT2RAKE,_IMPRACE,_CHISPNC,_CRACE1,CAGEG,_CLLCPWT,_DUALUSE,_DUALCOR,_LLCPWT2,_LLCPWT,_RFHLTH,_PHYS14D,_MENT14D,_HLTHPL2,_HCVU654,_TOTINDA,_EXTETH3,_ALTETH3,_DENVST3,_MICHD,_LTASTH1,_CASTHM1,_ASTHMS1,_DRDXAR2,_MRACE1,_HISPANC,_RACE,_RACEG21,_RACEGR3,_RACEPRV,_SEX,_AGEG5YR,_AGE65YR,_AGE80,_AGE_G,HTIN4,HTM4,WTKG3,_BMI5,_BMI5CAT,_RFBMI5,_CHLDCNT,_EDUCAG,_INCOMG1,_RFMAM23,_MAM402Y,_CRVSCRN,_RFPAP37,_HPV5YR1,_PAPHPV1,_HADCOLN,_CLNSCP2,_HADSIGM,_SGMSCP2,_SGMS102,_RFBLDS6,_STOLDN2,_VIRCOL2,_SBONTI2,_CRCREC3,_SMOKER3,_RFSMOK3,_CURECI3,LCSLAST_,LCSNUMC_,_LCSAGE,_LCSYSMK,_PACKDAY,_PACKYRS,_LCSYQTS,_LCSSMKG,_LCSELIG,_LCSCTSN,_LCSPSTF,DRNKANY6,DROCDY4_,_RFBING6,_DRNKWK3,_RFDRHV9,_FLSHOT7,_PNEUMO3,_AIDTST4
0,1.0,2.0,2282024,2.0,28.0,2024,1100.0,2024000001,2.024000e+09,1.0,1.0,NaN,1.0,2.0,1.0,1.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,3.0,2.0,88.0,88.0,3.0,2.0,2.0,1.0,1.0,1.0,1.0,2.0,2.0,0,2.0,NaN,1.0,2.0,2.0,2.0,2.0,1.0,3.0,NaN,3.0,4.0,1.0,2.0,NaN,1.0,2.0,7.0,88.0,99.0,NaN,131.0,504.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,1.0,2.0,1.0,2.0,1.0,1.0,1.0,3.0,NaN,NaN,1.0,1.0,7.0,1.0,7.0,2.0,NaN,NaN,2.0,NaN,3.0,1.0,NaN,NaN,1.0,7.0,NaN,888.0,NaN,NaN,NaN,1.0,777777.0,1.0,2.0,2.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,5.0,2.0,2.0,5.0,2.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,10.0,1.0,NaN,NaN,1.0,1.0,1.0,11011.0,28.393219,1.0,28.393219,1.0,NaN,NaN,NaN,NaN,1.0,0.465294,220.149005,261.525511,1.0,2.0,1.0,1.0,9.0,1.0,2.0,1.0,1.0,2.0,1.0,1.0,3.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,2.0,12.0,2.0,78.0,6.0,64.0,163.0,5942.0,2249.0,2.0,1.0,1.0,2.0,9.0,1.0,NaN,1.0,NaN,NaN,NaN,1.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,1.0,1.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN,6.0,2.0,NaN,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,2.0,2.0


In [3]:
# Manual feature selection - the goal is to drop features such as administrative and survey design variables.
# These variables tell how the data was collected, not what respondents reported about their health or behaviour.

df.drop(columns=[
    # When the survey was conducted and general identifiers whether is eligible to be included in the survey
    "FMONTH", "IDATE", "IMONTH", "IDAY", "IYEAR", "DISPCODE", "SEQNO", "_PSU", "SAFETIME",
    "LADULT1", "CADULT1", "RESPSLC1", "CTELENM1", "CTELNUM1","CELPHON1", "CELLFON5", "LANDLINE",
    "LANDSEX3", "CELLSEX3",
    
    # Residency and household variables
    "PVTRESD1", "PVTRESD3", "COLGHOUS", "STATERE1", "CSTATE1", "NUMADULT", 
    "HHADULT", "NUMHHOL4", "NUMPHON4", "CPDEMO1C", "CCLGHOUS", "GUNLOAD", 
    "LOADULK2", "FIREARM5", "_CHLDCNT", "CHILDREN", "CAGEG", "RENTHOM1",

    # Survey language and version variables
    "QSTVER", "QSTLANG", "ICFQSTVR",

    # Statistical sampling weights & design variables
    "_STSTR", "_STRWT", "_RAWRAKE", "_WT2RAKE", "_CLLCPWT", 
    "_DUALUSE", "_DUALCOR", "_LLCPWT2", "_LLCPWT",

    # Post Course of Treatment questions
    "CSRVINST", "CSRVINSR", "CSRVDEIN", "PCSTALK2", "CSRVSUM",

    # Caregiver - about the respondent's relationship to their caregiver, 
    # not about the respondent's health or behaviour.
    "CAREGIV1", "CRGVREL5", "CRGVPRB4", "CRGVALZD", 
    "CRGVNURS", "CRGVPER2", "CRGVHOU2", "CRGVHRS2", "CRGVLNG2", 

    # Family Planning
    "RCSRLTN2", "HADSEX", "PFPPRVN4", "TYPCNTR9", "NOBCUSE8",

    "RCSXBRTH", "RCSGEND1", "HPVDSHT", "CSRVCTL2",
], inplace=True)

print("Dataset after dropping columns that are unrelated to health outcomes by design:", df.shape)
df.head(1)

Dataset after dropping columns that are unrelated to health outcomes by design: (456218, 229)


,_STATE,SEXVAR,GENHLTH,PHYSHLTH,MENTHLTH,POORHLTH,PRIMINS2,PERSDOC3,MEDCOST1,CHECKUP1,EXERANY2,LASTDEN4,RMVTETH4,CVDINFR4,CVDCRHD4,CVDSTRK3,ASTHMA3,ASTHNOW,CHCSCNC1,CHCOCNC1,CHCCOPD3,ADDEPEV3,CHCKDNY2,HAVARTH4,DIABETE4,DIABAGE4,MARITAL,EDUCA,VETERAN3,EMPLOY1,INCOME3,PREGNANT,WEIGHT2,HEIGHT3,DEAF,BLIND,DECIDE,DIFFWALK,DIFFDRES,DIFFALON,HADMAM,HOWLONG,CERVSCRN,CRVCLCNC,CRVCLPAP,CRVCLHPV,HADHYST2,HADSIGM4,COLNSIGM,COLNTES1,SIGMTES1,LASTSIG4,COLNCNCR,VIRCOLO1,VCLNTES2,SMALSTOL,STOLTEST,STOOLDN2,BLDSTFIT,SDNATES1,SMOKE100,SMOKDAY2,USENOW3,ECIGNOW3,LCSFIRST,LCSNUMCG,LCSCTSC1,LCSSCNCR,LCSCTWHN,ALCDAY4,AVEDRNK4,DRNK3GE5,MAXDRNKS,FLUSHOT7,FLSHTMY3,IMFVPLA5,PNEUVAC4,HIVTST7,HIVTSTD3,HIVRISK5,PDIABTS1,PREDIAB2,DIABTYPE,INSULIN1,CHKHEMO3,EYEEXAM1,DIABEYE1,DIABEDU1,FEETSORE,ARTHEXER,SHINGLE2,HPVADVC4,HPVADSH1,TETANUS1,CNCRDIFF,CNCRAGE,CNCRTYP2,CSRVTRT3,CSRVDOC1,CSRVRTRN,CSRVCLIN,CSRVPAIN,PSATEST1,PSATIME1,PCPSARS2,PSASUGS1,CIMEMLO1,CDWORRY,CDDISCU1,CDHOUS1,CDSOCIA1,ACEDEPRS,ACEDRINK,ACEDRUGS,ACEPRISN,ACEDIVRC,ACEPUNCH,ACEHURT1,ACESWEAR,ACETOUCH,ACETTHEM,ACEHVSEX,ACEADSAF,ACEADNED,LSATISFY,EMTSUPRT,SDLONELY,SDHEMPLY,FOODSTMP,SDHFOOD1,SDHBILLS,SDHUTILS,SDHTRNSP,HOWSAFE1,MARIJAN1,MARJSMOK,MARJEAT,MARJVAPE,MARJDAB,MARJOTHR,USEMRJN4,LASTSMK2,STOPSMK2,MENTCIGS,MENTECIG,HEATTBCO,SSBSUGR2,SSBFRUT3,CASTHDX2,CASTHNO2,SOMALE,SOFEMALE,_METSTAT,_URBSTAT,MSCODE,_IMPRACE,_CHISPNC,_CRACE1,_RFHLTH,_PHYS14D,_MENT14D,_HLTHPL2,_HCVU654,_TOTINDA,_EXTETH3,_ALTETH3,_DENVST3,_MICHD,_LTASTH1,_CASTHM1,_ASTHMS1,_DRDXAR2,_MRACE1,_HISPANC,_RACE,_RACEG21,_RACEGR3,_RACEPRV,_SEX,_AGEG5YR,_AGE65YR,_AGE80,_AGE_G,HTIN4,HTM4,WTKG3,_BMI5,_BMI5CAT,_RFBMI5,_EDUCAG,_INCOMG1,_RFMAM23,_MAM402Y,_CRVSCRN,_RFPAP37,_HPV5YR1,_PAPHPV1,_HADCOLN,_CLNSCP2,_HADSIGM,_SGMSCP2,_SGMS102,_RFBLDS6,_STOLDN2,_VIRCOL2,_SBONTI2,_CRCREC3,_SMOKER3,_RFSMOK3,_CURECI3,LCSLAST_,LCSNUMC_,_LCSAGE,_LCSYSMK,_PACKDAY,_PACKYRS,_LCSYQTS,_LCSSMKG,_LCSELIG,_LCSCTSN,_LCSPSTF,DRNKANY6,DROCDY4_,_RFBING6,_DRNKWK3,_RFDRHV9,_FLSHOT7,_PNEUMO3,_AIDTST4
0,1.0,2.0,3.0,2.0,88.0,88.0,3.0,2.0,2.0,1.0,1.0,1.0,1.0,2.0,2.0,0,2.0,NaN,1.0,2.0,2.0,2.0,2.0,1.0,3.0,NaN,3.0,4.0,2.0,7.0,99.0,NaN,131.0,504.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,1.0,2.0,1.0,2.0,1.0,1.0,1.0,3.0,NaN,NaN,1.0,1.0,7.0,1.0,7.0,2.0,NaN,NaN,2.0,NaN,3.0,1.0,NaN,NaN,1.0,7.0,NaN,888.0,NaN,NaN,NaN,1.0,777777.0,1.0,2.0,2.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,5.0,2.0,2.0,5.0,2.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,2.0,1.0,1.0,1.0,1.0,NaN,NaN,1.0,2.0,1.0,1.0,9.0,1.0,2.0,1.0,1.0,2.0,1.0,1.0,3.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,2.0,12.0,2.0,78.0,6.0,64.0,163.0,5942.0,2249.0,2.0,1.0,2.0,9.0,1.0,NaN,1.0,NaN,NaN,NaN,1.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,1.0,1.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN,6.0,2.0,NaN,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,2.0,2.0


In [4]:
# Find highly correlated features using Cramér's V for categorical variables.

# pairs_df = find_high_correlation_pairs(df, threshold=0.85, sample_n = 50_000, random_state = 42)
# print(pairs_df.to_string(index=False))

In [5]:
# Remove redundant features

df.drop(columns=[
    # General health / days unhealthy
    # Keep: GENHLTH, PHYSHLTH, MENTHLTH, _TOTINDA 
    # Precise amount of days unhealthy in the past 30 days, while the alternatives are coarser categories or binary variables
    "_RFHLTH", "_PHYS14D", "_MENT14D", "POORHLTH", "EXERANY2",

    # BMI / anthropometrics
    # Keep: _BMI5 (continuous computed BMI)
    # Raw height and weight in all unit encodings are redundant as we have _BMI5. 
    # _BMI5CAT and _RFBMI5 are redundant to _BMI5.
    "WEIGHT2", "HEIGHT3", "HTIN4", "HTM4", 
    "WTKG3", "_BMI5CAT", "_RFBMI5",

    # Race / ethnicity
    # Keep: _RACE (8-category combined variable covering all race variants)
    "_RACEG21", "_RACEGR3", "_RACEPRV", "_MRACE1", 
    "_IMPRACE", "_CHISPNC", "_CRACE1", "_HISPANC",

    # Age
    # Keep: _AGE80 (Alternatives are encoded in age groups, but _AGE80 is a continuous number 
    # that gives us more precise information about the respondent's age)
    "_AGE65YR", "_AGE_G", "_AGEG5YR", "_LCSAGE",

    # Smoking
    # Keep: _SMOKER3 (Derived variable that combines current / former / never / every day smoking status)
    # Keep: LCSNUMC_ (preciese number of cigarettes smoked per day)
    # Raw inputs SMOKE100 and SMOKDAY2 are fully absorbed by _SMOKER3.
    # _RFSMOK3 is a binary collapse of _SMOKER3. USENOW3, LASTSMK2,
    "_RFSMOK3", "USENOW3", "LASTSMK2", "STOPSMK2", "LCSNUMCG",
    "HEATTBCO", "SMOKE100", "SMOKDAY2", "_PACKDAY", "_LCSSMKG",
    "_PACKYRS",

    # E-cigarettes / vaping
    # Keep: ECIGNOW3 (Has all 3-category current / former / never status)
    "_CURECI3", "MENTCIGS", "MENTECIG",
    
    # Marijuana / cannabis
    # Keep: MARIJAN1 (Give us not just the status but precise amount of consumption days)
    "MARJEAT", "MARJDAB", "MARJOTHR", "MARJSMOK", "MARJVAPE", "USEMRJN4",

    # Alcohol
    # Keep: _DRNKWK3 (Calculated total number of alcoholic beverages consumed per week, 
    # while the alternatives are coarser categories or binary variables)
    "_RFBING6", "DRNKANY6", "ALCDAY4", "DRNK3GE5", 
    "MAXDRNKS", "DROCDY4_", "_RFDRHV9", "AVEDRNK4",

    # Education / Income
    # Keep: EDUCA / INCOME3 (EDUCA and INCOME3 respectively provide a more detailed range of education and income levels)
    "_EDUCAG", "_INCOMG1", 

    # Health insurance / access
    # Keep: PRIMINS2 , PERSDOC3, CHECKUP1
    "MEDCOST1", "_HLTHPL2" , "_HCVU654",

    # Sex
    # Keep: _SEX
    "SEXVAR",

    # Asthma
    # Keep: _ASTHMS1 (Computed Asthma Status, covers all 3 current, former and never asthma status)
    "_LTASTH1", "_CASTHM1", "CASTHDX2", "CASTHNO2", "ASTHNOW", "ASTHMA3", 

    # Cancer / Lung cancer screening test
    # Keep: _LCSCTSN, CERVSCRN, 
    "CSRVDOC1", "CSRVRTRN", "CSRVCLIN", "LCSSCNCR", "LCSCTSC1", "_LCSPSTF",
    "LCSCTWHN", 

    # Dental / oral health
    # Keep: LASTDEN4, RMVTETH4
    "_EXTETH3", "_DENVST3", "_ALTETH3",

    # PSA test follow-ups
    # Keep: PSATEST1 (ever had PSA test)
    "PSATIME1", "PCPSARS2", "PSASUGS1",

    # Mammography / cervical screening
    # Keep: HOWLONG
    "_SGMS102", "_SGMSCP2", "_CLNSCP2", "_MAM402Y", "_RFMAM23",
    "_CRVSCRN", "_RFPAP37",

    # Flu shot
    # FLUSHOT7 (as it has no age restriction, while _FLSHOT7 is only for 65+)
    "_FLSHOT7", "_PNEUMO3", "FLSHTMY3", "IMFVPLA5", "HIVTSTD3",

    # Colorectal screening
    # Keep: COLNSIGM (handles both sigmoidoscopy and colonoscopy, as well as both)
    "_HADSIGM", "_HADCOLN", "VIRCOLO1", "_VIRCOL2", "_RFBLDS6", "_STOLDN2",
    "STOOLDN2", "_CRCREC3", "BLDSTFIT", "SDNATES1", "VCLNTES2", "SMALSTOL",
    "_SBONTI2", "HADSIGM4", "LASTSIG4",
    
    # Arthritis
    # Keep: HAVARTH4
    "_DRDXAR2", "ARTHEXER", "CHKHEMO3", 

    # Urbanicity
    # Keep: _URBSTAT
    "_METSTAT", "MSCODE",

    # HIV/AIDS
    # Keep: HIVTST7
    "_AIDTST4", "_HPV5YR1", "_PAPHPV1", "HPVADSH1",

    # Angina or Coronary Heart Disease
    # Keep: _MICHD and CVDINFR4
    "CVDCRHD4",
], inplace=True)

print("Dataset after dropping redundant features:", df.shape)
df.head(1)

Dataset after dropping redundant features: (456218, 115)


,_STATE,GENHLTH,PHYSHLTH,MENTHLTH,PRIMINS2,PERSDOC3,CHECKUP1,LASTDEN4,RMVTETH4,CVDINFR4,CVDSTRK3,CHCSCNC1,CHCOCNC1,CHCCOPD3,ADDEPEV3,CHCKDNY2,HAVARTH4,DIABETE4,DIABAGE4,MARITAL,EDUCA,VETERAN3,EMPLOY1,INCOME3,PREGNANT,DEAF,BLIND,DECIDE,DIFFWALK,DIFFDRES,DIFFALON,HADMAM,HOWLONG,CERVSCRN,CRVCLCNC,CRVCLPAP,CRVCLHPV,HADHYST2,COLNSIGM,COLNTES1,SIGMTES1,COLNCNCR,STOLTEST,ECIGNOW3,LCSFIRST,FLUSHOT7,PNEUVAC4,HIVTST7,HIVRISK5,PDIABTS1,PREDIAB2,DIABTYPE,INSULIN1,EYEEXAM1,DIABEYE1,DIABEDU1,FEETSORE,SHINGLE2,HPVADVC4,TETANUS1,CNCRDIFF,CNCRAGE,CNCRTYP2,CSRVTRT3,CSRVPAIN,PSATEST1,CIMEMLO1,CDWORRY,CDDISCU1,CDHOUS1,CDSOCIA1,ACEDEPRS,ACEDRINK,ACEDRUGS,ACEPRISN,ACEDIVRC,ACEPUNCH,ACEHURT1,ACESWEAR,ACETOUCH,ACETTHEM,ACEHVSEX,ACEADSAF,ACEADNED,LSATISFY,EMTSUPRT,SDLONELY,SDHEMPLY,FOODSTMP,SDHFOOD1,SDHBILLS,SDHUTILS,SDHTRNSP,HOWSAFE1,MARIJAN1,SSBSUGR2,SSBFRUT3,SOMALE,SOFEMALE,_URBSTAT,_TOTINDA,_MICHD,_ASTHMS1,_RACE,_SEX,_AGE80,_BMI5,_SMOKER3,LCSLAST_,LCSNUMC_,_LCSYSMK,_LCSYQTS,_LCSELIG,_LCSCTSN,_DRNKWK3
0,1.0,3.0,2.0,88.0,3.0,2.0,1.0,1.0,1.0,2.0,0,1.0,2.0,2.0,2.0,2.0,1.0,3.0,NaN,3.0,4.0,2.0,7.0,99.0,NaN,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,1.0,2.0,1.0,2.0,1.0,1.0,3.0,NaN,1.0,7.0,1.0,NaN,1.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,5.0,2.0,2.0,5.0,2.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,2.0,1.0,1.0,2.0,3.0,1.0,2.0,78.0,2249.0,4.0,NaN,NaN,NaN,NaN,2.0,NaN,5.397605e-79


In [6]:
# Compare the results of correlation analysis after removing redundant features
# from feature_analysis import find_high_correlation_pairs

# pairs_df = find_high_correlation_pairs(df, threshold=0.80, sample_n = 50_000, random_state = 42)
# print(pairs_df.to_string(index=False))

In [7]:
# Features categorised per the Overleaf report tables
# Table 1 (tab:stroke_feature_rationale) — included features
included_features = [
    # Demographic
    '_STATE', '_SEX', '_RACE', '_AGE80',
    'EDUCA', 'INCOME3', 'MARITAL', 'EMPLOY1', 'VETERAN3', 'PREGNANT',
    '_URBSTAT',

    # General health
    'GENHLTH', 'PHYSHLTH', 'MENTHLTH',

    # Lifestyle / behavioural
    '_TOTINDA', '_BMI5',
    '_SMOKER3', 'LCSFIRST', 'LCSLAST_', 'LCSNUMC_', '_LCSYSMK', '_LCSYQTS',
    '_LCSELIG', '_LCSCTSN',
    'ECIGNOW3', 'MARIJAN1',
    '_DRNKWK3',
    'SSBSUGR2', 'SSBFRUT3',

    # Diabetes
    'DIABETE4', 'DIABAGE4', 'DIABTYPE', 'PDIABTS1', 'PREDIAB2',
    'INSULIN1', 'EYEEXAM1', 'DIABEYE1', 'DIABEDU1', 'FEETSORE',

    # Clinical comorbidities
    '_ASTHMS1', 'CHCCOPD3', 'CHCKDNY2', 'HAVARTH4', 'ADDEPEV3',
    'CHCOCNC1', 'CNCRTYP2', 'CSRVTRT3', 'CNCRAGE', 'CSRVPAIN',

    # Healthcare access
    'PRIMINS2', 'PERSDOC3', 'LASTDEN4', 'RMVTETH4',

    # Female-specific
    'HADHYST2',

    # Adverse childhood experiences (ACEs)
    'ACEDEPRS', 'ACEDRINK', 'ACEDRUGS', 'ACEPRISN', 'ACEDIVRC',
    'ACEPUNCH', 'ACEHURT1', 'ACESWEAR',
    'ACETOUCH', 'ACETTHEM', 'ACEHVSEX',
    'ACEADSAF', 'ACEADNED',

    # Social / psychosocial
    'LSATISFY', 'EMTSUPRT', 'SDLONELY',
]

# Table 2 (tab:stroke_feature_exclusion_rationale) — excluded features
excluded_features = [
    # Healthcare engagement proxies with no independent stroke signal
    'CHECKUP1',
    'HADMAM', 'HOWLONG',
    'CERVSCRN', 'CRVCLCNC', 'CRVCLPAP', 'CRVCLHPV',
    'COLNSIGM', 'COLNTES1', 'SIGMTES1', 'COLNCNCR', 'STOLTEST',
    'HIVTST7', 'HIVRISK5',
    'PSATEST1',
    # Vaccines — no causal stroke risk
    'SHINGLE2', 'HPVADVC4', 'TETANUS1',
    # Social-needs / material-hardship — redundant with income/education
    'FOODSTMP', 'SDHFOOD1', 'SDHBILLS', 'SDHUTILS', 'SDHTRNSP',
    # Weak / non-specific cancer
    'CHCSCNC1',
]

df.drop(columns=excluded_features, inplace=True)
print("Dataset after dropping excluded features:", df.shape)

# Table 3 (tab:stroke_feature_uncertainty_rationale) — uncertain features
uncertain_features = [
    # Post-stroke functional limitations (temporal ambiguity)
    'DIFFWALK', 'DIFFDRES', 'DIFFALON',
    # Cognitive items (pre- vs post-stroke unclear)
    'CIMEMLO1', 'CDWORRY', 'CDDISCU1', 'CDHOUS1', 'CDSOCIA1', 'DECIDE',
    # Social-needs items with directional ambiguity
    'SDHEMPLY',
    # Vaccinations with uncertain directionality
    'FLUSHOT7', 'PNEUVAC4',
    # Sensory / neighbourhood items
    'DEAF', 'BLIND', 'HOWSAFE1',
    # Sexual orientation
    'SOMALE', 'SOFEMALE',
    # Cancer count (likely redundant)
    'CNCRDIFF',
    # CVD variables (overlap between _MICHD, CVDINFR4, CVDCRHD4)
    '_MICHD', 'CVDINFR4', 'CVDCRHD4',
]

Dataset after dropping excluded features: (456218, 91)


In [ ]:
combined_cols = [column for column in df.columns if column != target_col]

# Shared cols (both for lifestyle and clinical)
shared_cols = ['_SEX']

# Lifestyle-only cols
lifestyle_cols = ['_TOTINDA']

clinical_only_cols = [
    c for c in combined_cols
    if c not in set(lifestyle_cols + shared_cols)
]

# Two variants: with and without uncertain features
uncertain_set = set(uncertain_features)

def drop_uncertain(cols):
    """Return cols with uncertain features removed."""
    return [c for c in cols if c not in uncertain_set]

dataset_definitions = {
    'with_uncertain': {
        'lifestyle': lifestyle_cols + shared_cols,
        'clinical': clinical_only_cols + shared_cols,
        'combined': combined_cols,
    },
    'without_uncertain': {
        'lifestyle': drop_uncertain(lifestyle_cols) + drop_uncertain(shared_cols),
        'clinical': drop_uncertain(clinical_only_cols) + drop_uncertain(shared_cols),
        'combined': drop_uncertain(combined_cols),
    },
}

# Register datasets
datasets = {}

for uncertainty_variant, feature_sets in dataset_definitions.items():
    for feat_name, feat_cols in feature_sets.items():
        # Only keep cols that actually exist in df
        valid_cols = [c for c in feat_cols if c in df.columns]
        ds = df[valid_cols + [target_col]]

        dataset_id = f"{feat_name}_{uncertainty_variant}"
        datasets[dataset_id] = {
            'df': ds,
            'feature_set': feat_name,
            'uncertainty': uncertainty_variant,
        }
        register_dataset(dataset_id, ds, label=dataset_id.replace('_', ' ').title())
        print(f"Registered dataset: {dataset_id}  features={ds.shape[1] - 1}")

print(f"\nTotal datasets: {len(datasets)}")

# Algorithms
algorithms = {
    'random_forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'xgboost': XGBClassifier(eval_metric='logloss', random_state=42),
    'lightgbm': LGBMClassifier(random_state=42, verbose=-1),
}

# Train: 3 algorithms * 6 datasets = 18 models
for dataset_id, dataset_info in datasets.items():
    ds = dataset_info['df']
    feat_name = dataset_info['feature_set']
    uncertainty = dataset_info['uncertainty']

    X, y = ds.drop(columns=[target_col]), ds[target_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    for algo_name, clf in algorithms.items():
        model_id = f"{algo_name}_{dataset_id}"
        print(f"Training {model_id}...")

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_prob = clf.predict_proba(X_test)[:, 1]

        pkl_path = (MODELS_DIR / f"{model_id}.pkl").resolve()
        joblib.dump(clf, pkl_path)

        report_raw = classification_report(y_test, y_pred, output_dict=True)
        cm = confusion_matrix(y_test, y_pred)
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        importances = getattr(clf, 'feature_importances_', np.zeros(len(X.columns)))

        register_model(
            model_id=model_id,
            algorithm=algo_name,
            feature_set=feat_name,
            uncertainty_variant=uncertainty,
            model_path=str(pkl_path),
            metrics={
                'auc': roc_auc_score(y_test, y_prob),
                'accuracy': accuracy_score(y_test, y_pred),
                'f1': f1_score(y_test, y_pred),
                'precision': precision_score(y_test, y_pred),
                'recall': recall_score(y_test, y_pred),
            },
            classification_report={
                'classes': {
                    k: v for k, v in report_raw.items()
                    if k not in ('accuracy', 'macro avg', 'weighted avg')
                },
                'macro_avg': report_raw['macro avg'],
                'weighted_avg': report_raw['weighted avg'],
                'accuracy': report_raw['accuracy'],
            },
            confusion_matrix={
                'tn': int(cm[0, 0]), 'fp': int(cm[0, 1]),
                'fn': int(cm[1, 0]), 'tp': int(cm[1, 1]),
            },
            feature_importances=[
                {'feature': col, 'importance': float(imp)}
                for col, imp in zip(X.columns, importances)
            ],
            roc_curve={'fpr': fpr.tolist(), 'tpr': tpr.tolist()},
            feature_columns=list(X.columns),
        )
        print(f"Registered {model_id}")

print(f"\nDone. {len(datasets) * len(algorithms)} models trained and registered.")

Registered dataset: lifestyle_with_uncertain  features=2
Registered dataset: clinical_with_uncertain  features=89
Registered dataset: combined_with_uncertain  features=90
Registered dataset: lifestyle_without_uncertain  features=2
Registered dataset: clinical_without_uncertain  features=69
Registered dataset: combined_without_uncertain  features=70

Total datasets: 6
Training random_forest_lifestyle_with_uncertain...


c:\Users\anton\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anton\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anton\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anton\anaconda3\Lib

Registered random_forest_lifestyle_with_uncertain
Training xgboost_lifestyle_with_uncertain...


c:\Users\anton\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anton\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anton\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anton\anaconda3\Lib

Registered xgboost_lifestyle_with_uncertain
Training lightgbm_lifestyle_with_uncertain...


c:\Users\anton\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anton\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anton\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anton\anaconda3\Lib

Registered lightgbm_lifestyle_with_uncertain
Training random_forest_clinical_with_uncertain...
